# Customer Feedback & Sales Analytics

Цель анализа - найти продукты, каналы и причины обратной связи, которые одновременно влияют на клиентский опыт и финансовый результат.

## 1. Загрузка данных

Используется очищенная выгрузка с расчетными полями: прибыль, маржинальность, признак проблемной обратной связи и NPS-группа.

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/sales_feedback_clean.csv', parse_dates=['order_date'])
df.head()

 order_id order_date customer_id          region channel       manager    category          product  quantity  discount  revenue   cost  rating   feedback_reason  response_time_hours  is_return   month  profit  margin_pct  has_problem rating_group nps_group
    10001 2026-01-01      C01719          Москва телефон      Ким В.В. Компрессоры Компрессор А-120         1      0.00   149099 110274       4     срок поставки                   13          0 2026-01   38825       0.260         True      средняя   passive
    10002 2026-01-01      C01409     Новосибирск    сайт Кузнецов И.В.     Датчики      Датчик D-17         2      0.03    37027  23401       3 неполное описание                   17          0 2026-01   13626       0.368         True       низкая detractor
    10003 2026-01-01      C00788 Санкт-Петербург    сайт  Новиков П.К.      Насосы       Насос Н-45         1      0.07    62447  46813       5      нет проблемы                    7          0 2026-01   15634       0.250     

## 2. Контроль структуры и качества

Перед расчетом KPI проверяются размер выборки, период, дубликаты заказов и пропуски.

In [2]:
quality = pd.DataFrame({
    'metric': ['rows', 'columns', 'duplicated_order_id', 'missing_values', 'period_start', 'period_end'],
    'value': [
        len(df),
        df.shape[1],
        int(df['order_id'].duplicated().sum()),
        int(df.isna().sum().sum()),
        df['order_date'].min().date().isoformat(),
        df['order_date'].max().date().isoformat(),
    ],
})
quality

             metric      value
               rows       4995
            columns         22
duplicated_order_id          0
     missing_values          0
       period_start 2026-01-01
         period_end 2026-06-30

## 3. Executive KPI

Базовые показатели нужны, чтобы связать клиентскую обратную связь с управленческими метриками.

In [3]:
summary = pd.DataFrame([{
    'orders': len(df),
    'customers': df['customer_id'].nunique(),
    'revenue': int(df['revenue'].sum()),
    'profit': int(df['profit'].sum()),
    'margin_pct': round(df['profit'].sum() / df['revenue'].sum(), 3),
    'avg_rating': round(df['rating'].mean(), 2),
    'problem_share': round(df['has_problem'].mean(), 3),
    'avg_response_hours': round(df['response_time_hours'].mean(), 1),
}])
summary

 orders  customers   revenue    profit  margin_pct  avg_rating  problem_share  avg_response_hours
   4995       1903 635500827 170472394       0.268        4.19           0.29                14.6

## 4. Продукты и категории в зоне риска

Продуктовый анализ ранжирует товары не только по жалобам, но и по выручке, маржинальности и средней оценке.

In [4]:
product_metrics = (
    df.groupby(['category', 'product'], as_index=False)
      .agg(
          orders=('order_id', 'count'),
          revenue=('revenue', 'sum'),
          profit=('profit', 'sum'),
          avg_rating=('rating', 'mean'),
          problem_share=('has_problem', 'mean'),
          avg_response_hours=('response_time_hours', 'mean'),
          return_share=('is_return', 'mean'),
      )
)
product_metrics['margin_pct'] = (product_metrics['profit'] / product_metrics['revenue']).round(3)
product_metrics.sort_values(['problem_share', 'revenue'], ascending=[False, False]).head(8)

        category               product  orders   revenue   profit  avg_rating  problem_share  avg_response_hours  return_share  margin_pct
       Редукторы          Редуктор R-8     762 109816358 24175846    4.072178       0.370079           15.254593      0.059055       0.220
     Компрессоры      Компрессор А-120     957 222571551 62223596    4.099269       0.366771           15.571578      0.056426       0.280
     Контроллеры        Контроллер C-9     433  43249378 14693531    4.159353       0.307159           15.489607      0.034642       0.340
         Датчики           Датчик D-17     588  17768496  6418752    4.144558       0.284014           14.239796      0.035714       0.361
         Фильтры           Фильтр F-30     476  20030953  6210439    4.260504       0.256303           14.239496      0.033613       0.310
         Клапаны           Клапан V-12     431  25848011  7511960    4.301624       0.225058           14.067285      0.025522       0.291
          Насосы           

## 5. Каналы продаж

Канальный срез показывает, где операционная скорость ответа может ухудшать оценку клиента.

In [5]:
channel_metrics = (
    df.groupby('channel', as_index=False)
      .agg(
          orders=('order_id', 'count'),
          revenue=('revenue', 'sum'),
          avg_rating=('rating', 'mean'),
          problem_share=('has_problem', 'mean'),
          avg_response_hours=('response_time_hours', 'mean'),
      )
      .sort_values('avg_response_hours', ascending=False)
)
channel_metrics

          channel  orders   revenue  avg_rating  problem_share  avg_response_hours
            почта    1275 160482682    4.138039       0.332549           21.010980
повторная продажа     995 131930161    4.191960       0.295477           12.778894
             сайт    1706 216611733    4.202227       0.273154           12.395662
          телефон    1019 126476251    4.220805       0.260059           12.231600

## 6. Причины негативной обратной связи

Причины оцениваются через количество проблемных заказов и выручку в зоне риска.

In [6]:
issue_metrics = (
    df[df['has_problem']]
      .groupby('feedback_reason', as_index=False)
      .agg(
          problem_orders=('order_id', 'count'),
          revenue_at_risk=('revenue', 'sum'),
          avg_rating=('rating', 'mean'),
      )
      .sort_values('problem_orders', ascending=False)
)
issue_metrics

      feedback_reason  problem_orders  revenue_at_risk  avg_rating
        срок поставки             262         44132886    3.263359
            документы             194         28213162    3.164948
         комплектация             193         22379405    3.305699
                 цена             176         30239669    3.267045
    неполное описание             174         18663010    3.258621
        совместимость             171         18104297    3.058480
             доставка             142         16344082    3.246479
качество консультации             137         17700917    3.226277

## 7. Проверка гипотезы о времени ответа

Гипотеза: чем дольше первый ответ менеджера, тем ниже оценка клиента и выше доля проблемных заказов.

In [7]:
response_bins = pd.cut(
    df['response_time_hours'],
    bins=[0, 12, 24, 10_000],
    labels=['<=12h', '13-24h', '>24h'],
)
response_analysis = (
    df.assign(response_bucket=response_bins)
      .groupby('response_bucket', observed=True, as_index=False)
      .agg(
          orders=('order_id', 'count'),
          avg_rating=('rating', 'mean'),
          problem_share=('has_problem', 'mean'),
      )
)
response_analysis

response_bucket  orders  avg_rating  problem_share
          <=12h    2327    4.526859       0.045982
         13-24h    2045    4.035697       0.386308
           >24h     623    3.418941       0.886035

## 8. Матрица приоритизации

Приоритет повышается, если продукт одновременно дает большую выручку, имеет высокую долю проблем и низкую среднюю оценку.

In [8]:
priority = product_metrics.copy()
priority['priority_score'] = (
    priority['revenue'].rank(pct=True) * 0.4 +
    priority['problem_share'].rank(pct=True) * 0.4 +
    (1 - priority['avg_rating'].rank(pct=True)) * 0.2
).round(3)
priority = priority.sort_values('priority_score', ascending=False)[
    ['product', 'category', 'revenue', 'avg_rating', 'problem_share', 'margin_pct', 'priority_score']
]
priority.head(5)

              product         category   revenue  avg_rating  problem_share  margin_pct  priority_score
     Компрессор А-120      Компрессоры 222571551    4.099269       0.366771       0.280           0.900
         Редуктор R-8        Редукторы 109816358    4.072178       0.370079       0.220           0.875
       Контроллер C-9      Контроллеры  43249378    4.159353       0.307159       0.340           0.600
          Датчик D-17          Датчики  17768496    4.144558       0.284014       0.361           0.425
Электродвигатель E-55 Электродвигатели 110962987    4.291525       0.215254       0.260           0.425

## 9. Вывод

Фокус улучшений стоит выбирать по связке `выручка + клиентский риск + управляемость причины`. В первую очередь нужно контролировать продукты с высокой долей проблемных заказов, ускорять первый ответ по почтовому каналу и регулярно выносить матрицу приоритетов в управленческий отчет.